# Track 1, Step 2: VCF Pre-Filtering

This notebook implements Step 2 of the Variant Prediction pipeline. The goal is to strip out sequencing artifacts and massive amounts of noise from the raw ~5 million variant VCF (`data/WGS_EX2312012_HGWCNDSX7.vcf.gz`).

### Process Overview:
1. **Filter 1 (Quality Control & Allelic Balance):** We enforce strict thresholds based on the machine's confidence and biological likelihood:
   - `FILTER`: Must be `PASS` or missing.
   - `GQ` (Genotype Quality): >= 20 (99% confidence).
   - `DP` (Depth of Coverage): >= 10.
   - `AB` (Allelic Balance): For heterozygous calls, the alternate allele must represent 30% to 70% of the total reads to rule out contamination or mapping artifacts.

2. **Output Generation:** Rather than saving an intermediate `.vcf.gz` file, we directly output a flattened, structured Pandas DataFrame as a CSV file (`data/track1_step2_intrinsic_qc.csv`). This perfectly aligns with the required downstream format for Step 3 Annotation, integrating all original VCF fields with our newly calculated QC metrics.

*Note on Filter 2 (Population Rarity):* As per project discussions, the allele-aware common variant subtraction (gnomAD < 0.01%) requires local VCF databases. This step is computationally decoupled from this notebook to avoid API rate-limiting on 2.6+ million variants and will be handled during the annotation step.*

In [1]:
import os
import cyvcf2 # cyvcf2 is used because it's a fast C-based wrapper for VCF parsing, which is essential when iterating over 5 million rows.
import time # time is used to track how long the filtering process takes.
import pandas as pd

# Define and verify data paths. 
# Why: Ensuring the files exist before starting prevents the script from failing halfway through.
input_vcf = "../data/WGS_EX2312012_HGWCNDSX7.vcf.gz"
output_csv = "../data/track1_step2_intrinsic_qc.csv"

assert os.path.exists(input_vcf), f"File not found: {input_vcf}"
print(f"Data file found: {input_vcf}")


Data file found: ../data/WGS_EX2312012_HGWCNDSX7.vcf.gz


## Filter 1: Quality Control & Allelic Balance (Removing "Typos")
We process the VCF line by line using `cyvcf2` to identify variants that meet our strict sequencing thresholds. 

### How is `AB_FILTER` calculated?
**Allelic Balance (AB)** is mathematically found by dividing the number of reads supporting the alternate allele (`FORMAT_AD_ALT`) by the total number of reads spanning that specific position (`FORMAT_AD_REF` + `FORMAT_AD_ALT`).
*   **Why we need it:** In a true biological heterozygous mutation (one mutated copy from one parent, one healthy copy from the other), you would expect roughly 50% of the sequencer's reads to show the reference sequence and 50% to show the mutation. If the Allelic Balance is highly skewed (e.g., only 10% of reads show the alternate allele), it strongly implies a sequencer mapping artifact or sample contamination rather than a real mutation. We strictly require AB to be between 0.30 and 0.70 for heterozygous variants.

During the iteration, variants that pass all checks will be batched into chunks and appended directly to the final `CSV` file to preserve system memory.

In [2]:
# Open the input VCF for reading.
vcf_in = cyvcf2.VCF(input_vcf)

start_time = time.time()

# Trackers for filtering statistics
stats = {
    'total_variants': 0,
    'dropped_filter': 0,
    'dropped_missing_metrics': 0,
    'dropped_gq': 0,
    'dropped_dp': 0,
    'dropped_gt': 0,
    'dropped_ab': 0,
    'passed_variants': 0
}

variant_records = []
chunk_size = 250000
first_chunk = True

# Iterate through every variant in the VCF file.
for variant in vcf_in:
    stats['total_variants'] += 1
    
    # 1. Filter out variants that failed the sequencer's internal quality checks.
    if variant.FILTER is not None and variant.FILTER != "PASS":
        stats['dropped_filter'] += 1
        continue
        
    # Extract formats array for our sample.
    gq_array = variant.format('GQ')
    dp_array = variant.format('DP')
    ad_array = variant.format('AD') # [REF_depth, ALT_depth, ...]
    
    # Skip if any of these critical metrics are missing.
    if gq_array is None or dp_array is None or ad_array is None:
        stats['dropped_missing_metrics'] += 1
        continue
        
    gq = gq_array[0][0]
    dp = dp_array[0][0]
    
    # 2. Apply hard depth and quality cutoffs.
    if gq < 20:
        stats['dropped_gq'] += 1
        continue
    if dp < 10:
        stats['dropped_dp'] += 1
        continue
        
    # 3. Check Genotype (GT).
    gt_type = variant.gt_types[0]
    if gt_type == 0 or gt_type == 3: # Ignore HOM_REF and UNKNOWN
        stats['dropped_gt'] += 1
        continue
        
    # 4. Calculate Allelic Balance (AB).
    ref_depth = ad_array[0][0]
    alt_depth = ad_array[0][1] if len(ad_array[0]) > 1 else 0
    calc_dp = ref_depth + alt_depth
    
    if calc_dp == 0:
        stats['dropped_ab'] += 1
        continue
        
    ab = alt_depth / calc_dp
    
    # 5. Filter based on Allelic Balance.
    if gt_type == 1: # Heterozygous (expected 50% REF, 50% ALT)
        if ab < 0.30 or ab > 0.70:
            stats['dropped_ab'] += 1
            continue
    elif gt_type == 2: # Homozygous ALT (expected ~100% ALT)
        if ab < 0.90:
            stats['dropped_ab'] += 1
            continue
            
    # 6. If surviving, format into structured tabular record
    record = {
        'CHROM': variant.CHROM,
        'POS': variant.POS,
        'ID': variant.ID if variant.ID else '.',
        'REF': variant.REF,
        'ALT': ",".join(variant.ALT) if variant.ALT else ".",
        'QUAL': variant.QUAL,
        'FILTER': variant.FILTER if variant.FILTER else 'PASS',
        
        # Extracted INFO fields
        'INFO_DP': variant.INFO.get('DP'),
        'INFO_MQ': variant.INFO.get('MQ'),
        'INFO_FS': variant.INFO.get('FS'),
        
        # Extracted FORMAT fields
        'FORMAT_GT': gt_type,
        'FORMAT_DP': dp,
        'FORMAT_GQ': gq,
        'FORMAT_AD_REF': ref_depth,
        'FORMAT_AD_ALT': alt_depth,
        
        # --- NEW COLUMNS ---
        'AB_FILTER': round(ab, 3)
    }
    variant_records.append(record)
    stats['passed_variants'] += 1
    
    # Memory management: Write chunk to CSV and clear the list
    if len(variant_records) >= chunk_size:
        df_chunk = pd.DataFrame(variant_records)
        df_chunk.to_csv(output_csv, mode='w' if first_chunk else 'a', header=first_chunk, index=False)
        first_chunk = False
        variant_records = []

# Write any remaining variants
if variant_records:
    df_chunk = pd.DataFrame(variant_records)
    df_chunk.to_csv(output_csv, mode='w' if first_chunk else 'a', header=first_chunk, index=False)
    variant_records = []

vcf_in.close()

end_time = time.time()
print(f"Processed {stats['total_variants']:,} total variants in {end_time - start_time:.2f} seconds.")
print(f"\n--- FILTERING STATISTICS ---")
print(f"Dropped by 'PASS' flag:      {stats['dropped_filter']:,}")
print(f"Dropped missing metrics:     {stats['dropped_missing_metrics']:,}")
print(f"Dropped by GQ < 20:          {stats['dropped_gq']:,}")
print(f"Dropped by DP < 10:          {stats['dropped_dp']:,}")
print(f"Dropped by GT (Ref/Unk):     {stats['dropped_gt']:,}")
print(f"Dropped by Allelic Balance:  {stats['dropped_ab']:,}")
print(f"----------------------------")
print(f"Variants surviving Intrinsic QC: {stats['passed_variants']:,} ({(stats['passed_variants']/stats['total_variants'])*100:.1f}%)")
print(f"Structured CSV saved directly to: {output_csv}")


Processed 5,012,204 total variants in 57.43 seconds.

--- FILTERING STATISTICS ---
Dropped by 'PASS' flag:      271,414
Dropped missing metrics:     0
Dropped by GQ < 20:          29,727
Dropped by DP < 10:          34,646
Dropped by GT (Ref/Unk):     1,784,768
Dropped by Allelic Balance:  246,584
----------------------------
Variants surviving Intrinsic QC: 2,645,065 (52.8%)
Structured CSV saved directly to: ../data/track1_step2_intrinsic_qc.csv


## Reviewing the Final Output
Let's open the resulting CSV file using Pandas to verify the final tabular structure and ensure the `AB_FILTER` metric correctly populated alongside the intrinsic columns.

In [3]:
# Read the final saved CSV back into memory (preview the top 1000 rows for speed)
df_final = pd.read_csv(output_csv, nrows=1000)

print(f"Successfully loaded preview of {output_csv}")
print(f"Dataset contains {df_final.shape[1]} columns.")
display(df_final.head())


Successfully loaded preview of ../data/track1_step2_intrinsic_qc.csv
Dataset contains 16 columns.


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO_DP,INFO_MQ,INFO_FS,FORMAT_GT,FORMAT_DP,FORMAT_GQ,FORMAT_AD_REF,FORMAT_AD_ALT,AB_FILTER
0,1,16298,rs200451305,C,T,446.769989,PASS,41,40.840000,18.659,1,41,99,24,17,0.415
1,1,98921,rs199533981,AG,A,468.730011,PASS,26,43.080002,6.668,1,26,99,8,18,0.692
2,1,99166,rs78627600,C,T,505.769989,PASS,47,44.330002,0.000,1,47,99,32,15,0.319
3,1,99180,rs201323924,T,C,487.769989,PASS,46,43.880001,0.000,1,46,99,31,15,0.326
4,1,102951,rs200298631,C,T,614.770020,PASS,41,41.450001,1.216,1,41,99,17,24,0.585
